# 🏛️ Crypto Institutional Quant Fund - Cloud Training Hub
### نموذج التعلم العميق الهجين: LightGBM + PyTorch Time-Series Transformer
---
تم تصميم هذه المفكرة لتعمل بسلاسة وخفة تامة على **Google Colab**:
- **معالجة سحابية 100%:** يتم التدريب على كرت T4 في خوادم جوجل دون استهلاك موارد جهازك.
- **تجاوز تلقائي للحظر الجغرافي (Binance 451 Fix):** ضبط فوري لخادم Binance Vision المفتوح.
- **إخراج هادئ ونظيف (Quiet Output):** تقليل السجلات المزعجة لمنع تجميد المتصفح.

### 1️⃣ فحص كرت الشاشة (GPU Verification)
تأكد أن بيئة التشغيل مفعلة على كرت T4:
`Runtime` -> `Change runtime type` -> **T4 GPU**.

In [ ]:
!nvidia-smi -L

### 2️⃣ تجهيز ملفات المشروع (تلقائي بنقرة واحدة)
إذا قمت بسحب وإفلات ملف **`Crypto-Fund-Isolated.zip`** في لوحة الملفات (📁 يسار الشاشة)، سيقوم هذا السطر بفكه فوراً وضبط المسار.

In [ ]:
import os, shutil
from pathlib import Path

%cd /content
if os.path.exists('/content/Crypto-Fund-Isolated.zip'):
    !unzip -o -q /content/Crypto-Fund-Isolated.zip -d /content/Crypto-Institutional-Quant-Fund
    print('✅ تم فك ضغط ملف المشروع بنجاح!')
elif not os.path.exists('/content/Crypto-Institutional-Quant-Fund'):
    print('⚠️ يُرجى سحب وإفلات ملف Crypto-Fund-Isolated.zip في لوحة الملفات على اليسار ثم إعادة تشغيل هذه الخلية.')

%cd /content/Crypto-Institutional-Quant-Fund
print('📂 مسار العمل الحالي:', os.getcwd())

### 3️⃣ تثبيت الحزم الرياضية ومكتبات التعلم العميق (FreqAI & PyTorch)

In [ ]:
# تثبيت سريع وخفيف لـ Freqtrade و PyTorch و LightGBM بدون مخرجات مزعجة
!pip install -q --no-cache-dir freqtrade[freqai,pytorch] lightgbm joblib
print('✅ اكتمل تثبيت مكتبات الذكاء الاصطناعي بنجاح!')

### 4️⃣ تأكيد إعدادات الحماية وتجاوز حظر Binance الجغرافي (Auto-Patch)
يضمن هذا السطر ربط خادم Binance Vision المفتوح وحصر الاستعلامات على Spot فقط.

In [ ]:
import json

config_file = Path('user_data/config_spot_hybrid.json')
if config_file.exists():
    with open(config_file, 'r') as f:
        cfg = json.load(f)
    
    # تثبيت إعداد تجاوز 451 الرسمي
    spot_only_mirror = {
        'options': {'fetchMarkets': ['spot']},
        'urls': {
            'api': {
                'public': 'https://data-api.binance.vision/api/v3',
                'private': 'https://data-api.binance.vision/api/v3'
            }
        }
    }
    cfg['exchange']['ccxt_config'] = spot_only_mirror
    cfg['exchange']['ccxt_async_config'] = spot_only_mirror
    
    with open(config_file, 'w') as f:
        json.dump(cfg, f, indent=4)
    print('✅ إعدادات Binance Vision جاهزة ومحصورة في Spot 100%!')
else:
    print('❌ ملف config_spot_hybrid.json غير موجود، تأكد من تشغيل الخطوة 2.')

### 5️⃣ بدء تدريب النموذج الهجين والباك تيست (Training & Walk-Forward)
يعمل التدريب الآن على كرت الشاشة T4 مع دمج LightGBM وشبكة الترانسفورمر المالي.

In [ ]:
# تشغيل التدريب مع التحكم في مستوى السجلات ليكون الأداء سلساً وخفيفاً
!freqtrade backtesting \
    --config user_data/config_spot_hybrid.json \
    --strategy InstitutionalDualStrategy \
    --freqaimodel HybridTreeTransformer \
    --timerange 20260916-20261101

### 6️⃣ تصدير النموذج المدرب وحفظه في Google Drive

In [ ]:
from google.colab import drive
import shutil
from pathlib import Path

# ربط Google Drive
drive.mount('/content/drive')

model_src = Path('user_data/models/spot_hybrid_transformer_model')
zip_dest = '/content/drive/MyDrive/spot_hybrid_transformer_model'

if model_src.exists():
    shutil.make_archive(zip_dest, 'zip', model_src)
    print(f'🎉 تم حفظ النموذج المدرب بنجاح داخل Google Drive: {zip_dest}.zip')
    print('👉 يمكنك الآن تحميل هذا الملف ووضعه في مجلد user_data/models على جهازك لتشغيل التداول فوراً!')
else:
    print('⚠️ مجلد النموذج لم يتم إنشاؤه بعد. تأكد من إتمام الخطوة 5 بنجاح.')